# Monte Carlo Data Quality — Exploration Notebook

This notebook demonstrates the core DQ engine interactively.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.core.simulation import MonteCarloEngine, QualityDimension
from src.core.profiler import StatisticalProfiler
from src.core.detectors import AnomalyDetector

sns.set_theme(style='whitegrid')
print('Imports OK')

## 1. Load seed data

In [ ]:
orders = pd.read_csv('../data/samples/orders.csv', parse_dates=['created_at', 'updated_at'])
baseline = pd.read_csv('../data/samples/orders_baseline.csv', parse_dates=['created_at', 'updated_at'])
customers = pd.read_csv('../data/samples/customers.csv', parse_dates=['created_at'])
print(f'orders: {orders.shape}, baseline: {baseline.shape}, customers: {customers.shape}')
orders.head()

## 2. Monte Carlo DQ Simulation

In [ ]:
engine = MonteCarloEngine(n_simulations=2000, random_seed=42)
result = engine.run(
    orders,
    key_columns=['order_id'],
    timestamp_col='created_at',
    max_age_hours=9999,
)

print(f'Overall score : {result.overall_score:.4f}')
print(f'Passed        : {result.passed}')
print(f'Threshold     : {result.threshold}')
print()
for d in result.dimensions:
    print(f'  {d.dimension:15s}  mean={d.mean_score:.4f}  CI=[{d.p5:.4f}, {d.p95:.4f}]  p={d.p_value:.4f}')

In [ ]:
fig, axes = plt.subplots(1, len(result.dimensions), figsize=(14, 4))
for ax, dim in zip(axes, result.dimensions):
    ax.bar(['Score'], [dim.mean_score], color='steelblue', alpha=0.8)
    ax.errorbar(['Score'], [dim.mean_score],
                yerr=[[dim.mean_score - dim.p5], [dim.p95 - dim.mean_score]],
                fmt='none', color='black', capsize=6)
    ax.set_ylim(0, 1.05)
    ax.set_title(dim.dimension.capitalize())
    ax.axhline(result.threshold, color='red', linestyle='--', linewidth=1, label='threshold')
fig.suptitle('DQ Dimension Scores with 90% CI', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Statistical Profiling

In [ ]:
profiler = StatisticalProfiler()
profile = profiler.profile(orders)

cols_df = pd.DataFrame([
    {
        'column': c.name,
        'dtype': c.dtype,
        'null_rate': c.null_rate,
        'cardinality': c.cardinality,
        'mean': c.mean,
        'std': c.std,
    }
    for c in profile.columns
])
cols_df

## 4. Drift Detection

In [ ]:
drift_reports = profiler.detect_drift(baseline, orders)

drift_df = pd.DataFrame([
    {
        'column': r.column,
        'ks_statistic': r.ks_statistic,
        'ks_p_value': r.ks_p_value,
        'js_divergence': r.js_divergence,
        'drift_detected': r.drift_detected,
    }
    for r in drift_reports
])
drift_df.style.applymap(
    lambda v: 'background-color: #ffcccc' if v is True else '',
    subset=['drift_detected']
)

## 5. Anomaly Detection

In [ ]:
detector = AnomalyDetector()

zscore_reports = detector.detect_zscore(orders, columns=['amount_usd'])
iqr_reports = detector.detect_iqr(orders, columns=['amount_usd'])
iso_report = detector.detect_isolation_forest(orders, columns=['amount_usd'])

print(f'Z-score anomalies  : {zscore_reports[0].n_anomalies} ({zscore_reports[0].anomaly_rate:.2%})')
print(f'IQR anomalies      : {iqr_reports[0].n_anomalies} ({iqr_reports[0].anomaly_rate:.2%})')
print(f'Isolation Forest   : {iso_report.n_anomalies} ({iso_report.anomaly_rate:.2%})')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
clean = orders['amount_usd'].dropna()
ax.hist(clean, bins=80, color='steelblue', alpha=0.7, label='All values')

outlier_idx = zscore_reports[0].anomaly_indices
if outlier_idx:
    ax.hist(orders.loc[outlier_idx, 'amount_usd'].dropna(), bins=20,
            color='red', alpha=0.9, label='Z-score outliers')

ax.set_xlabel('amount_usd')
ax.set_ylabel('Count')
ax.set_title('Order Amount Distribution with Detected Anomalies')
ax.legend()
plt.tight_layout()
plt.show()